# [3장 4강] - Tensor로 Self-Attention 미니 구현 (1)

<aside>
🎯

**실습 목표**

- Tensor 연산만으로 Self-Attention 전체 흐름을 구현합니다.
- `nn.Module`로 Q·K·V projection을 등록하고 재사용합니다.
- Batch와 Padding Mask를 함께 처리하는 모듈로 확장합니다.
</aside>

<aside>
🧑‍💻

기존처럼 `scores = None` 한 줄을 채우지 않습니다. 함수와 클래스의 입력 검사부터 반환값까지 직접 설계하세요.

</aside>

---

## 핵심 보조 실습. Self-Attention 함수 전체 구현

### 시작 코드

```python
import torch

torch.manual_seed(42)
X = torch.randn(5, 8)
W_q = torch.randn(8, 8)
W_k = torch.randn(8, 8)
W_v = torch.randn(8, 8)
```

### 수행해야 할 작업

1. `self_attention(X, W_q, W_k, W_v)` 함수를 작성하세요.
2. Q, K, V projection, scaling, softmax, Value 가중합을 순서대로 구현하세요.
3. output과 weights를 함께 반환하세요.
4. output이 입력과 같은 `[T, H]`, weights가 `[T, T]`인지 검증하세요.

    
   **해설**
    
  Self-Attention은 입력 token 수를 유지하면서 각 token 표현에 다른 위치의 정보를 섞습니다. Output shape가 입력과 같다고 값까지 같은 것은 아닙니다.

In [1]:
import math
import torch

torch.manual_seed(42)
X = torch.randn(5, 8)
W_q = torch.randn(8, 8)
W_k = torch.randn(8, 8)
W_v = torch.randn(8, 8)


def self_attention(X, W_q, W_k, W_v):
    hidden_size = X.size(-1)
    for weight in (W_q, W_k, W_v):
        if weight.shape != (hidden_size, hidden_size):
            raise ValueError("Projection weight shape가 입력 hidden size와 맞지 않습니다.")

    Q = X @ W_q
    K = X @ W_k
    V = X @ W_v

    scores = (Q @ K.T) / math.sqrt(Q.size(-1))
    weights = torch.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights


output, weights = self_attention(X, W_q, W_k, W_v)
print("output:", tuple(output.shape))
print("weights:", tuple(weights.shape))

assert output.shape == X.shape
assert weights.shape == (5, 5)
assert torch.allclose(weights.sum(dim=-1), torch.ones(5))

output: (5, 8)
weights: (5, 5)


## 핵심 실습. 학습 가능한 SelfAttention 모듈 작성

### 시작 코드

```python
import torch
from torch import nn

torch.manual_seed(7)
X = torch.randn(2, 4, 6)
```

### 수행해야 할 작업

1. `SelfAttention(nn.Module)` 클래스를 작성하세요.
2. 생성자에서 `q_proj`, `k_proj`, `v_proj`, `out_proj`를 등록하세요.
3. Forward에서 `[B, T, H]`를 처리하고 output과 weights를 반환하세요.
4. 모든 Linear가 `model.parameters()`에 포함되는지 확인하세요.

    
  **해설**
    
  Weight를 함수 밖의 임의 Tensor로 두는 대신 `nn.Linear`로 등록하면 optimizer가 해당 parameter를 찾고 갱신할 수 있습니다. `out_proj`는 attention context를 다시 hidden space에서 조합합니다.

In [2]:
import math
import torch
from torch import nn

torch.manual_seed(7)
X = torch.randn(2, 4, 6)


class SelfAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x):
        if x.ndim != 3 or x.size(-1) != self.hidden_size:
            raise ValueError("입력은 [B, T, H]이고 H가 hidden_size와 같아야 합니다.")

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.hidden_size)
        weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(weights, V)
        return self.out_proj(context), weights


model = SelfAttention(hidden_size=6)
output, weights = model(X)
print(tuple(output.shape), tuple(weights.shape))
print("parameter 수:", sum(p.numel() for p in model.parameters()))

assert output.shape == (2, 4, 6)
assert weights.shape == (2, 4, 4)
assert sum(p.numel() for p in model.parameters()) == 144

(2, 4, 6) (2, 4, 4)
parameter 수: 144


## 참고·심화 실습. Padding Mask를 지원하는 모듈로 확장

### 시작 코드

```python
X = torch.randn(2, 5, 6)
attention_mask = torch.tensor([
    [1, 1, 1, 1, 0],
    [1, 1, 1, 0, 0],
], dtype=torch.bool)
```

### 수행해야 할 작업

1. 앞의 클래스를 `MaskedSelfAttention`으로 확장하세요.
2. `attention_mask=None`도 허용하세요.
3. Mask가 있으면 Key padding 열의 score를 softmax 전에 가리세요.
4. Padding 열의 weight가 0인지 검증하세요.
    
  **해설**
    
  여기서는 Padding 위치가 Key로 선택되지 않게 했습니다. Padding Query의 output까지 제거해야 하는 모델이라면 forward 마지막에 query mask를 곱하는 처리를 추가할 수 있습니다.

In [3]:
import math
import torch
from torch import nn

torch.manual_seed(7)
X = torch.randn(2, 5, 6)
attention_mask = torch.tensor([[1, 1, 1, 1, 0], [1, 1, 1, 0, 0]], dtype=torch.bool)


class MaskedSelfAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x, attention_mask=None):
        Q, K, V = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.hidden_size)

        if attention_mask is not None:
            if attention_mask.shape != x.shape[:2]:
                raise ValueError("attention_mask는 [B, T]여야 합니다.")
            key_mask = attention_mask.to(torch.bool).unsqueeze(1)
            scores = scores.masked_fill(~key_mask, float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        output = self.out_proj(torch.matmul(weights, V))
        return output, weights


model = MaskedSelfAttention(6)
output, weights = model(X, attention_mask)
print(tuple(output.shape), tuple(weights.shape))

assert torch.allclose(weights[0, :, 4], torch.zeros(5))
assert torch.allclose(weights[1, :, 3:], torch.zeros(5, 2))

(2, 5, 6) (2, 5, 5)
